# DocuEntitySeek

Capability del progetto:

1. **Entity Discovery** — individua e colora entità direttamente nel testo;
2. **Field Extraction** — estrae informazioni strutturate e le mostra separatamente in un pannello.

Pipeline:

```text
PDF
 ↓
PyMuPDF
 ↓
testo pagina per pagina + mappa degli offset
 ↓
GLiNER2.5 long-context
 ├──────────────────────┐
 ↓                      ↓
Entity Discovery     Field Extraction
 ↓                      ↓
annotazioni NER      informazioni strutturate
nel documento        nella sidebar
```

# 1. Separare pipeline e schema di dominio

DocuEntitySeek sarà specializzato inizialmente in contratti, bandi, disciplinari e capitolati.

Il piano Base userà uno schema Procurement preconfigurato. Il piano Premium potrà creare entità e campi personalizzati e quindi adattare la stessa pipeline ad altri domini.

Principio fondamentale:

```text
PIPELINE = rimane uguale
SCHEMA   = può cambiare
```

# 2. Perché usiamo le API long-context di GLiNER2.5

Un documento di gara può contenere decine o centinaia di pagine. Le API `extract_entities_long()` e `extract_json_long()` dividono automaticamente il testo in chunk sovrapposti, analizzano tutto il documento, rimappano gli span agli offset globali e ricompongono le duplicazioni dovute all'overlap.

Il chunking viene misurato in **parole separate da whitespace**, non in caratteri.

Baseline DocuMind Procurement:

```text
chunk_size    = 512 parole
chunk_overlap = 96 parole
```

La documentazione GLiNER2 suggerisce `384/64` come punto di partenza generale e overlap più alti (`96–128`) quando informazioni importanti possono cadere vicino ai confini. Per contratti e bandi scegliamo `512/96` come compromesso iniziale fra contesto e costo di inferenza.

# 3. Due overlap diversi

## Chunk overlap
Serve a non perdere informazioni vicino al confine tra due chunk.

## Entity overlap
Riguarda invece due entità che occupano porzioni di testo sovrapposte, ad esempio:

```text
D.Lgs. 36/2023
██████████████   RIFERIMENTO_NORMATIVO
       ███████   CODICE
```

Nel backend conserviamo le detection. Nel viewer mostriamo una sola annotazione dominante quando due span si sovrappongono.

# 4. Regola semplice per le sovrapposizioni nel viewer

1. Duplicati con stessa label e stesso span → conserviamo la confidence più alta.
2. Per span sovrapposti ordiniamo per:
   - confidence più alta;
   - span più lungo;
   - posizione più a sinistra.
3. Accettiamo progressivamente soltanto span che non si sovrappongono a quelli già scelti.

Questa è una politica `flat` pensata per la **visualizzazione**. La Field Extraction rimane separata e non genera highlight nel testo.

# 5. Provenienza

Per ogni estrazione vogliamo poter risalire alla fonte. Conserveremo:

```text
text
label / field
confidence
global_start
global_end
pagina
page_start
page_end
```

GLiNER2.5 long-context restituisce gli offset globali. Noi aggiungiamo la mappa delle pagine del PDF.

# 6. Installazione

In [1]:
%pip install -U pymupdf pandas numpy "gliner2[local]"

  Using cached pymupdf-2.1-cp311-abi3-win_amd64.whl.metadata (26 kB)
  Using cached pandas-3.1.1-cp314-cp314-win_amd64.whl.metadata (16 kB)
  Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached gliner2-2.1.0-py3-none-any.whl.metadata (57 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached onnxruntime-1.31.1-cp314-cp314-win_amd64.whl.metadata (6.1 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached pymupdf_layout-2.1-py3-none-any.whl.metadata (657 bytes)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached pymupdf4llm-2.1-py3-none-any.whl.metadata (648 bytes)
  Using cached pydantic-2.14.3-py3-none-any.whl.metadata (141 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached peft-0.21.2-py3-none-any.whl.metada


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# 7. Import

In [2]:
from pathlib import Path
from typing import Any
import html
import json
import re
import time

import fitz
import pandas as pd

from IPython.display import HTML, display
from gliner2 import AutoExtractor

c:\Users\simon\OneDrive\Desktop\DocuMind\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\simon\OneDrive\Desktop\DocuMind\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 8. Configurazione tecnica

Tutti i parametri sono raccolti in un unico dizionario. In questo modo la pipeline rimane leggibile e il futuro backend potrà caricare la configurazione da un solo punto.

In [3]:
PIPELINE_CONFIG = {
    "model_id": "fastino/gliner2.5-multi-v1",
    "chunk_size": 512,
    "chunk_overlap": 96,
    "batch_size": 8,
    "entity_threshold": 0.70,
    "field_threshold": 0.50,
    "model_overlap_policy": "allow",
    "viewer_overlap_policy": "flat_confidence",
    "include_confidence": True,
    "include_spans": True,
}

pd.DataFrame([
    {"parametro": k, "valore": v}
    for k, v in PIPELINE_CONFIG.items()
])

,parametro,valore
0,model_id,fastino/gliner2.5-multi-v1
1,chunk_size,512
2,chunk_overlap,96
3,batch_size,8
4,entity_threshold,0.7
5,field_threshold,0.5
6,model_overlap_policy,allow
7,viewer_overlap_policy,flat_confidence
8,include_confidence,True
9,include_spans,True


# 9. Schema Procurement

Lo schema definisce **che cosa cercare**, non **come analizzare**. Ogni entity ha una descrizione e un colore per il viewer. Ogni field ha una descrizione precisa.

La lista è una baseline iniziale da verificare sui documenti reali.

In [4]:
PROCUREMENT_SCHEMA = {
    "name": "DocuMind Procurement",
    "entities": {
        "ORGANIZZAZIONE": {
            "description": "Enti pubblici, amministrazioni, aziende, società, operatori economici e altre organizzazioni nominate nel documento",
            "color": "#DCEBFF",
        },
        "PERSONA": {
            "description": "Nomi e cognomi di persone fisiche nominate nel documento",
            "color": "#FFE5D9",
        },
        "LOCALITA": {
            "description": "Città, comuni, province, regioni, nazioni, indirizzi e altri luoghi geografici",
            "color": "#E1F5E5",
        },
        "IMPORTO": {
            "description": "Importi monetari, prezzi, valori economici, cauzioni, garanzie e somme espresse con una valuta",
            "color": "#FFF1B8",
        },
        "DATA": {
            "description": "Date di calendario, scadenze, date di pubblicazione e altre date esplicitamente indicate",
            "color": "#E8DDF8",
        },
        "ORA": {
            "description": "Orari o ore specifiche esplicitamente indicate",
            "color": "#F7D9EE",
        },
        "RIFERIMENTO_NORMATIVO": {
            "description": "Leggi, decreti legislativi, regolamenti, articoli, commi e altri riferimenti normativi",
            "color": "#D9F0F0",
        },
        "IDENTIFICATIVO_GARA": {
            "description": "Codici identificativi della procedura come CIG, CUP, CPV e altri codici ufficiali di gara",
            "color": "#FCE0B2",
        },
        "CERTIFICAZIONE": {
            "description": "Certificazioni, attestazioni e standard richiesti o citati nel documento, come ISO o SOA",
            "color": "#E5E5FF",
        },
        "PRODOTTO_SERVIZIO": {
            "description": "Prodotti, servizi, forniture o categorie di prestazioni oggetto della procedura",
            "color": "#DFF3FF",
        },
    },
    "fields": {
        "stazione_appaltante": {"description": "Ente, amministrazione o organizzazione che indice o gestisce la procedura di gara", "dtype": "str"},
        "oggetto_gara": {"description": "Oggetto principale della procedura, cioè la fornitura, il servizio o il lavoro richiesto", "dtype": "str"},
        "cig": {"description": "Codice Identificativo Gara (CIG)", "dtype": "str"},
        "cup": {"description": "Codice Unico di Progetto (CUP), se presente", "dtype": "str"},
        "cpv": {"description": "Codice CPV principale della procedura, se presente", "dtype": "str"},
        "tipo_procedura": {"description": "Tipo di procedura di affidamento o gara, per esempio procedura aperta, negoziata o affidamento diretto", "dtype": "str"},
        "importo_complessivo": {"description": "Valore economico complessivo della procedura o dell'appalto", "dtype": "str"},
        "importo_base_asta": {"description": "Importo posto a base d'asta o base di gara", "dtype": "str"},
        "numero_lotti": {"description": "Numero complessivo dei lotti in cui è suddivisa la procedura", "dtype": "str"},
        "durata_contratto": {"description": "Durata prevista del contratto, appalto o servizio", "dtype": "str"},
        "scadenza_offerte": {"description": "Data e, se presente, ora entro cui devono essere presentate le offerte", "dtype": "str"},
        "termine_chiarimenti": {"description": "Data e, se presente, ora entro cui possono essere presentate richieste di chiarimento", "dtype": "str"},
        "criterio_aggiudicazione": {"description": "Criterio utilizzato per l'aggiudicazione, per esempio offerta economicamente più vantaggiosa o minor prezzo", "dtype": "str"},
        "responsabile_procedimento": {"description": "Nome del RUP o responsabile unico del progetto/procedimento", "dtype": "str"},
        "luogo_esecuzione": {"description": "Luogo principale previsto per l'esecuzione del contratto o della prestazione", "dtype": "str"},
    },
}

# 10. Esempio di schema Premium

La pipeline rimane identica; cambiano entità e campi. Questo esempio dimostra la flessibilità, ma non viene usato automaticamente.

In [5]:
EXAMPLE_PREMIUM_SCHEMA = {
    "name": "Real Estate Example",
    "entities": {
        "PERSONA": {"description": "Nomi di persone fisiche", "color": "#FFE5D9"},
        "IMMOBILE": {"description": "Immobili, edifici, appartamenti e unità immobiliari", "color": "#DCEBFF"},
        "IMPORTO": {"description": "Importi monetari e prezzi", "color": "#FFF1B8"},
    },
    "fields": {
        "prezzo_immobile": {"description": "Prezzo richiesto o concordato per l'immobile", "dtype": "str"},
        "indirizzo": {"description": "Indirizzo completo dell'immobile", "dtype": "str"},
    },
}

# 11. Schema attivo

In [6]:
ACTIVE_SCHEMA = PROCUREMENT_SCHEMA
print("Schema attivo:", ACTIVE_SCHEMA["name"])
print("Entità:", len(ACTIVE_SCHEMA["entities"]))
print("Campi:", len(ACTIVE_SCHEMA["fields"]))

Schema attivo: DocuMind Procurement
Entità: 10
Campi: 15


# 12. Path del PDF


In [8]:
PDF_PATH = Path(r"C:\Users\simon\OneDrive\Desktop\DocuMind\experimentation\DocuMind_Corpus_v1\corpus\disciplinare_1.pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(
        "Il file indicato in PDF_PATH non esiste. Modifica questa cella con il percorso reale."
    )
if PDF_PATH.suffix.lower() != ".pdf":
    raise ValueError("Il file selezionato non ha estensione .pdf")
print("Documento:", PDF_PATH)

Documento: C:\Users\simon\OneDrive\Desktop\DocuMind\experimentation\DocuMind_Corpus_v1\corpus\disciplinare_1.pdf


# 13. Lettura PDF e mappa delle pagine

PyMuPDF estrae il testo pagina per pagina. Per ogni pagina conserviamo `global_start` e `global_end`, riferiti alla stringa completa che invieremo a GLiNER.

In [9]:
PAGE_SEPARATOR = "\n\n"

def load_pdf_with_page_map(pdf_path: Path) -> dict:
    started = time.perf_counter()
    pdf = fitz.open(pdf_path)
    pages=[]; parts=[]; cursor=0
    for page_index in range(len(pdf)):
        page_number = page_index + 1
        page_text = pdf[page_index].get_text("text").strip()
        if parts:
            parts.append(PAGE_SEPARATOR)
            cursor += len(PAGE_SEPARATOR)
        global_start = cursor
        parts.append(page_text)
        cursor += len(page_text)
        global_end = cursor
        pages.append({
            "page": page_number,
            "text": page_text,
            "global_start": global_start,
            "global_end": global_end,
            "chars": len(page_text),
        })
    pdf.close()
    full_text = "".join(parts)
    return {
        "filename": pdf_path.name,
        "path": str(pdf_path),
        "page_count": len(pages),
        "pages": pages,
        "full_text": full_text,
        "chars": len(full_text),
        "seconds": time.perf_counter() - started,
    }

DOCUMENT = load_pdf_with_page_map(PDF_PATH)
print("File:", DOCUMENT["filename"])
print("Pagine:", DOCUMENT["page_count"])
print("Caratteri:", DOCUMENT["chars"])
print("Tempo:", round(DOCUMENT["seconds"], 3), "s")
if DOCUMENT["chars"] < 50:
    raise ValueError("È stato estratto troppo poco testo: il PDF potrebbe essere una scansione senza testo digitale.")

File: disciplinare_1.pdf
Pagine: 9
Caratteri: 36022
Tempo: 0.135 s


# 14. Controllo del testo estratto

In [10]:
pages_df = pd.DataFrame(DOCUMENT["pages"])[["page","chars","global_start","global_end"]].copy()
pages_df["preview"] = [re.sub(r"\s+", " ", p["text"])[:160] for p in DOCUMENT["pages"]]
display(pages_df.head(15))

,page,chars,global_start,global_end,preview
0,1,3263,0,3263,1 MINISTERO DELLE INFRASTRUTTURE E DEI TRASPOR...
1,2,4331,3265,7596,"2 Sono ammessi, altresì, i concorrenti aventi ..."
2,3,3253,7598,10851,3 conoscenza di elementi o circostanze tali da...
3,4,4126,10853,14979,"4 Procedura negoziata, di cui all’art. 36, let..."
4,5,5099,14981,20080,"5 1.1 Dichiarazioni di cui ai modelli A, B - m..."
5,6,4936,20082,25018,"6 dell’AVCP (ora ANAC), con le modificazioni a..."
6,7,3691,25020,28711,7 In caso di Consorzio (anche cooperativo) o R...
7,8,4602,28713,33315,8 c) verificare che più concorrenti non si sia...
8,9,2705,33317,36022,"9 Ai sensi dell’art. 13 del D.Lgs. n.196/2003,..."


# 15. Mapping span globale → pagina

GLiNER restituirà offset globali. Questa funzione li converte in pagina e offset locali alla pagina.

In [11]:
def map_global_span_to_pages(start: int | None, end: int | None, pages: list[dict]) -> dict:
    if start is None or end is None:
        return {"start_page":None,"end_page":None,"page_start":None,"page_end":None}
    overlaps=[]
    for page in pages:
        if start < page["global_end"] and end > page["global_start"]:
            overlaps.append(page)
    if not overlaps:
        return {"start_page":None,"end_page":None,"page_start":None,"page_end":None}
    first, last = overlaps[0], overlaps[-1]
    return {
        "start_page": first["page"],
        "end_page": last["page"],
        "page_start": start - first["global_start"],
        "page_end": (end - first["global_start"]) if first["page"] == last["page"] else None,
    }

# 16. Caricamento GLiNER2.5

In [12]:
started = time.perf_counter()
EXTRACTOR = AutoExtractor.from_pretrained(PIPELINE_CONFIG["model_id"])
MODEL_LOAD_SECONDS = time.perf_counter() - started
print("Modello:", PIPELINE_CONFIG["model_id"])
print("Caricato in:", round(MODEL_LOAD_SECONDS, 3), "s")

[transformers] You are using a model of type `gliner2` to instantiate a model of type `extractor`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.
[transformers] You are using a model of type `gliner2` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Modello: fastino/gliner2.5-multi-v1
Caricato in: 14.866 s


# 17. Preparazione Entity Types

Usiamo descrizioni, non solo label, per ridurre ambiguità su documenti lunghi e ripetitivi.

In [13]:
def build_entity_types(schema: dict) -> dict[str,str]:
    return {label: cfg["description"] for label, cfg in schema["entities"].items()}
ENTITY_TYPES = build_entity_types(ACTIVE_SCHEMA)
ENTITY_TYPES

{'ORGANIZZAZIONE': 'Enti pubblici, amministrazioni, aziende, società, operatori economici e altre organizzazioni nominate nel documento',
 'PERSONA': 'Nomi e cognomi di persone fisiche nominate nel documento',
 'LOCALITA': 'Città, comuni, province, regioni, nazioni, indirizzi e altri luoghi geografici',
 'IMPORTO': 'Importi monetari, prezzi, valori economici, cauzioni, garanzie e somme espresse con una valuta',
 'DATA': 'Date di calendario, scadenze, date di pubblicazione e altre date esplicitamente indicate',
 'ORA': 'Orari o ore specifiche esplicitamente indicate',
 'RIFERIMENTO_NORMATIVO': 'Leggi, decreti legislativi, regolamenti, articoli, commi e altri riferimenti normativi',
 'IDENTIFICATIVO_GARA': 'Codici identificativi della procedura come CIG, CUP, CPV e altri codici ufficiali di gara',
 'CERTIFICAZIONE': 'Certificazioni, attestazioni e standard richiesti o citati nel documento, come ISO o SOA',
 'PRODOTTO_SERVIZIO': 'Prodotti, servizi, forniture o categorie di prestazioni ogg

# 18. Entity Discovery long-context

Conserviamo confidence e span. `overlap_policy="allow"` mantiene le detection concorrenti; il viewer applicherà poi la sua regola flat.

In [14]:
def run_entity_discovery(document_text: str, entity_types: dict[str,str], config: dict = PIPELINE_CONFIG):
    started=time.perf_counter()
    result = EXTRACTOR.extract_entities_long(
        document_text,
        entity_types,
        threshold=config["entity_threshold"],
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        batch_size=config["batch_size"],
        include_spans=config["include_spans"],
        include_confidence=config["include_confidence"],
        overlap_policy=config["model_overlap_policy"],
    )
    return result, time.perf_counter()-started

RAW_ENTITY_RESULT, ENTITY_SECONDS = run_entity_discovery(DOCUMENT["full_text"], ENTITY_TYPES)
print("Entity Discovery:", round(ENTITY_SECONDS,3), "s")

Entity Discovery: 111.491 s


# 19. Normalizzazione NER

In [15]:
def normalize_entity_result(result: dict, document: dict) -> pd.DataFrame:
    rows=[]
    entities=result.get("entities", result)
    if not isinstance(entities, dict):
        return pd.DataFrame()
    for label, values in entities.items():
        if isinstance(values, (str,dict)): values=[values]
        if not isinstance(values, list): continue
        for value in values:
            if isinstance(value,str): item={"text":value,"confidence":None,"start":None,"end":None}
            elif isinstance(value,dict): item=value
            else: continue
            start=item.get("start"); end=item.get("end")
            rows.append({
                "label":label,"text":item.get("text"),"confidence":item.get("confidence"),
                "global_start":start,"global_end":end,
                **map_global_span_to_pages(start,end,document["pages"]),
            })
    if not rows:
        return pd.DataFrame(columns=["label","text","confidence","global_start","global_end","start_page","end_page","page_start","page_end"])
    return pd.DataFrame(rows).sort_values(["global_start","global_end","confidence"], ascending=[True,True,False], na_position="last").reset_index(drop=True)

RAW_ENTITIES = normalize_entity_result(RAW_ENTITY_RESULT, DOCUMENT)
print("NER grezze:", len(RAW_ENTITIES))
display(RAW_ENTITIES.head(30))

NER grezze: 212


,label,text,confidence,global_start,global_end,start_page,end_page,page_start,page_end
0,ORGANIZZAZIONE,MINISTERO DELLE INFRASTRUTTURE E DEI TRASPORTI,0.939665,7,53,1,1,7,53.0
1,LOCALITA,Veneto,0.793858,108,114,1,1,108,114.0
2,LOCALITA,Venezia,0.968386,146,153,1,1,146,153.0
3,LOCALITA,VENEZIA,0.939923,199,206,1,1,199,206.0
4,PERSONA,Cinzia Zincone,0.761565,262,276,1,1,262,276.0
5,RIFERIMENTO_NORMATIVO,legge l’11 \nsettembre 2020 n. 120,0.759606,559,592,1,1,559,592.0
6,RIFERIMENTO_NORMATIVO,art. 63,0.746273,669,676,1,1,669,676.0
7,LOCALITA,Venezia,0.912611,899,906,1,1,899,906.0
8,IDENTIFICATIVO_GARA,CIG 84126312B8,0.931170,908,922,1,1,908,922.0
9,IDENTIFICATIVO_GARA,CUP D75I14000050001,0.786799,925,944,1,1,925,944.0


# 20. Verifica span NER

In [16]:
def validate_global_spans(df: pd.DataFrame, full_text: str) -> pd.DataFrame:
    rows=[]
    for _, r in df.iterrows():
        s,e=r["global_start"],r["global_end"]
        valid=None if pd.isna(s) or pd.isna(e) else full_text[int(s):int(e)] == r["text"]
        rows.append({"label":r["label"],"text":r["text"],"page":r["start_page"],"span_valid":valid})
    return pd.DataFrame(rows)
ENTITY_SPAN_CHECK = validate_global_spans(RAW_ENTITIES, DOCUMENT["full_text"])
display(ENTITY_SPAN_CHECK)
if not ENTITY_SPAN_CHECK.empty and ENTITY_SPAN_CHECK["span_valid"].notna().any():
    print("Span validity:", round(float(ENTITY_SPAN_CHECK["span_valid"].dropna().mean()),4))

,label,text,page,span_valid
0,ORGANIZZAZIONE,MINISTERO DELLE INFRASTRUTTURE E DEI TRASPORTI,1,True
1,LOCALITA,Veneto,1,True
2,LOCALITA,Venezia,1,True
3,LOCALITA,VENEZIA,1,True
4,PERSONA,Cinzia Zincone,1,True
...,...,...,...,...
207,PERSONA,Francesco Sorrentino,9,True
208,PERSONA,FRANCESCO\nSORRENTINO,9,True
209,ORGANIZZAZIONE,MINISTERO DELLE\nINFRASTRUTTURE\nE DEI TRASPORTI,9,True
210,DATA,20.01.2021,9,True


Span validity: 1.0


# 21. Deduplica NER

In [17]:
def deduplicate_entities(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df.copy()
    w=df.copy(); w["_c"]=pd.to_numeric(w["confidence"],errors="coerce").fillna(-1.0)
    return (w.sort_values("_c",ascending=False)
             .drop_duplicates(["label","global_start","global_end"],keep="first")
             .drop(columns="_c")
             .sort_values(["global_start","global_end"])
             .reset_index(drop=True))
DEDUP_ENTITIES = deduplicate_entities(RAW_ENTITIES)
print("Prima:",len(RAW_ENTITIES),"Dopo:",len(DEDUP_ENTITIES))

Prima: 212 Dopo: 212


# 22. Risoluzione overlap per il viewer

In [18]:
def spans_overlap(sa:int,ea:int,sb:int,eb:int)->bool:
    return sa < eb and sb < ea

def resolve_overlaps_for_viewer(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df.copy()
    w=df.dropna(subset=["global_start","global_end"]).copy()
    w["_c"]=pd.to_numeric(w["confidence"],errors="coerce").fillna(-1.0)
    w["_len"]=w["global_end"]-w["global_start"]
    w=w.sort_values(["_c","_len","global_start"],ascending=[False,False,True])
    accepted=[]
    for _,c in w.iterrows():
        s,e=int(c["global_start"]),int(c["global_end"])
        if not any(spans_overlap(s,e,int(a["global_start"]),int(a["global_end"])) for a in accepted):
            accepted.append(c.to_dict())
    if not accepted: return df.iloc[0:0].copy()
    return (pd.DataFrame(accepted).drop(columns=["_c","_len"],errors="ignore")
            .sort_values(["global_start","global_end"]).reset_index(drop=True))

VIEWER_ENTITIES = resolve_overlaps_for_viewer(DEDUP_ENTITIES)
print("Entità deduplicate:",len(DEDUP_ENTITIES))
print("Annotazioni visibili:",len(VIEWER_ENTITIES))
print("Overlap nascosti:",len(DEDUP_ENTITIES)-len(VIEWER_ENTITIES))
display(VIEWER_ENTITIES.head(30))

Entità deduplicate: 212
Annotazioni visibili: 191
Overlap nascosti: 21


,label,text,confidence,global_start,global_end,start_page,end_page,page_start,page_end
0,ORGANIZZAZIONE,MINISTERO DELLE INFRASTRUTTURE E DEI TRASPORTI,0.939665,7,53,1,1,7,53.0
1,LOCALITA,Veneto,0.793858,108,114,1,1,108,114.0
2,LOCALITA,Venezia,0.968386,146,153,1,1,146,153.0
3,LOCALITA,VENEZIA,0.939923,199,206,1,1,199,206.0
4,PERSONA,Cinzia Zincone,0.761565,262,276,1,1,262,276.0
5,RIFERIMENTO_NORMATIVO,legge l’11 \nsettembre 2020 n. 120,0.759606,559,592,1,1,559,592.0
6,RIFERIMENTO_NORMATIVO,art. 63,0.746273,669,676,1,1,669,676.0
7,LOCALITA,Venezia,0.912611,899,906,1,1,899,906.0
8,IDENTIFICATIVO_GARA,CIG 84126312B8,0.931170,908,922,1,1,908,922.0
9,IDENTIFICATIVO_GARA,CUP D75I14000050001,0.786799,925,944,1,1,925,944.0


# 23. Conteggi per sidebar NER

In [19]:
entity_counts = (VIEWER_ENTITIES.groupby("label").size().reset_index(name="count").sort_values("count",ascending=False)
                 if not VIEWER_ENTITIES.empty else pd.DataFrame(columns=["label","count"]))
display(entity_counts)

,label,count
6,ORGANIZZAZIONE,55
9,RIFERIMENTO_NORMATIVO,53
7,PERSONA,24
3,IMPORTO,16
4,LOCALITA,12
2,IDENTIFICATIVO_GARA,10
1,DATA,8
8,PRODOTTO_SERVIZIO,6
5,ORA,4
0,CERTIFICAZIONE,3


# 24. Preparazione Structured IE

GLiNER2.5 accetta definizioni del tipo:

```text
cig::str::Codice Identificativo Gara
```

Aggiungere un field allo schema non richiede di modificare la pipeline.

In [20]:
def build_json_structure(schema: dict, structure_name: str="documento") -> dict[str,list[str]]:
    return {structure_name:[f"{name}::{cfg.get('dtype','str')}::{cfg['description']}" for name,cfg in schema["fields"].items()]}
JSON_STRUCTURE = build_json_structure(ACTIVE_SCHEMA)
JSON_STRUCTURE

{'documento': ['stazione_appaltante::str::Ente, amministrazione o organizzazione che indice o gestisce la procedura di gara',
  'oggetto_gara::str::Oggetto principale della procedura, cioè la fornitura, il servizio o il lavoro richiesto',
  'cig::str::Codice Identificativo Gara (CIG)',
  'cup::str::Codice Unico di Progetto (CUP), se presente',
  'cpv::str::Codice CPV principale della procedura, se presente',
  'tipo_procedura::str::Tipo di procedura di affidamento o gara, per esempio procedura aperta, negoziata o affidamento diretto',
  "importo_complessivo::str::Valore economico complessivo della procedura o dell'appalto",
  "importo_base_asta::str::Importo posto a base d'asta o base di gara",
  'numero_lotti::str::Numero complessivo dei lotti in cui è suddivisa la procedura',
  'durata_contratto::str::Durata prevista del contratto, appalto o servizio',
  'scadenza_offerte::str::Data e, se presente, ora entro cui devono essere presentate le offerte',
  'termine_chiarimenti::str::Data 

# 25. Field Extraction long-context

In [21]:
def run_field_extraction(document_text: str, json_structure: dict, config: dict=PIPELINE_CONFIG):
    started=time.perf_counter()
    result = EXTRACTOR.extract_json_long(
        document_text,
        json_structure,
        threshold=config["field_threshold"],
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        batch_size=config["batch_size"],
        include_spans=config["include_spans"],
        include_confidence=config["include_confidence"],
        overlap_policy=config["model_overlap_policy"],
    )
    return result, time.perf_counter()-started
RAW_FIELD_RESULT, FIELD_SECONDS = run_field_extraction(DOCUMENT["full_text"], JSON_STRUCTURE)
print("Field Extraction:",round(FIELD_SECONDS,3),"s")

Field Extraction: 125.09 s


# 26. Normalizzare valori Structured IE

In [22]:
def normalize_structured_value(value: Any) -> list[dict]:
    if value is None: return []
    if isinstance(value,str): return [{"text":value,"confidence":None,"start":None,"end":None}]
    if isinstance(value,dict):
        if "text" in value:
            return [{"text":value.get("text"),"confidence":value.get("confidence"),"start":value.get("start"),"end":value.get("end")}]
        rows=[]
        for child in value.values(): rows.extend(normalize_structured_value(child))
        return rows
    if isinstance(value,list):
        rows=[]
        for item in value: rows.extend(normalize_structured_value(item))
        return rows
    return []

# 27. Tabella dei candidati Field Extraction

In [23]:
def normalize_field_result(result: dict, schema: dict, document: dict, structure_name: str="documento") -> pd.DataFrame:
    rows=[]; records=result.get(structure_name,[])
    if isinstance(records,dict): records=[records]
    if not isinstance(records,list): records=[]
    known=set(schema["fields"].keys())
    for record_index,record in enumerate(records):
        if not isinstance(record,dict): continue
        for field_name,raw_value in record.items():
            if field_name not in known: continue
            for c in normalize_structured_value(raw_value):
                s,e=c.get("start"),c.get("end")
                rows.append({"field":field_name,"record_index":record_index,"text":c.get("text"),"confidence":c.get("confidence"),"global_start":s,"global_end":e,**map_global_span_to_pages(s,e,document["pages"])})
    if not rows:
        return pd.DataFrame(columns=["field","record_index","text","confidence","global_start","global_end","start_page","end_page","page_start","page_end"])
    return pd.DataFrame(rows).sort_values(["field","confidence"],ascending=[True,False],na_position="last").reset_index(drop=True)
FIELD_CANDIDATES = normalize_field_result(RAW_FIELD_RESULT, ACTIVE_SCHEMA, DOCUMENT)
print("Candidati:",len(FIELD_CANDIDATES))
display(FIELD_CANDIDATES)

Candidati: 450


,field,record_index,text,confidence,global_start,global_end,start_page,end_page,page_start,page_end
0,cig,4,84126312B8,6.653443e-01,11537,11547,4,4,684,694.0
1,cig,0,84126312B8,6.392978e-01,912,922,1,1,912,922.0
2,cig,6,CIG 84126312B8,4.816636e-01,11533,11547,4,4,680,694.0
3,cig,24,legale \nrappresentante dell’Impresa mandatari...,1.814885e-01,25337,25399,7,7,317,379.0
4,cig,26,Amministrazione,1.289190e-01,33086,33101,8,8,4373,4388.0
...,...,...,...,...,...,...,...,...,...,...
445,tipo_procedura,26,art. 76 del Codice a mezzo posta e/o fax o pos...,1.837958e-13,32825,33265,8,8,4112,4552.0
446,tipo_procedura,8,Amministrazione ad esito dell’esperimento dell...,1.280795e-18,12534,13492,4,4,1681,2639.0
447,tipo_procedura,1,l’esercizio 2020. \n10. I PAGAMENTI: avvengono...,6.634639e-21,2489,2983,1,1,2489,2983.0
448,tipo_procedura,29,art. 76 del Codice a mezzo posta e/o fax o pos...,2.153163e-21,32825,33667,8,9,4112,NaN


# 28. Deduplica dei candidati Field

In [24]:
def deduplicate_field_candidates(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty: return df.copy()
    w=df.copy(); w["_c"]=pd.to_numeric(w["confidence"],errors="coerce").fillna(-1.0)
    w["_text"]=w["text"].astype(str).str.lower().str.replace(r"\s+"," ",regex=True).str.strip()
    return (w.sort_values("_c",ascending=False)
             .drop_duplicates(["field","global_start","global_end","_text"],keep="first")
             .drop(columns=["_c","_text"])
             .sort_values(["field","confidence"],ascending=[True,False],na_position="last")
             .reset_index(drop=True))
FIELD_CANDIDATES = deduplicate_field_candidates(FIELD_CANDIDATES)
display(FIELD_CANDIDATES)

,field,record_index,text,confidence,global_start,global_end,start_page,end_page,page_start,page_end
0,cig,4,84126312B8,6.653443e-01,11537,11547,4,4,684,694.0
1,cig,0,84126312B8,6.392978e-01,912,922,1,1,912,922.0
2,cig,6,CIG 84126312B8,4.816636e-01,11533,11547,4,4,680,694.0
3,cig,24,legale \nrappresentante dell’Impresa mandatari...,1.814885e-01,25337,25399,7,7,317,379.0
4,cig,26,Amministrazione,1.289190e-01,33086,33101,8,8,4373,4388.0
...,...,...,...,...,...,...,...,...,...,...
442,tipo_procedura,26,art. 76 del Codice a mezzo posta e/o fax o pos...,1.837958e-13,32825,33265,8,8,4112,4552.0
443,tipo_procedura,8,Amministrazione ad esito dell’esperimento dell...,1.280795e-18,12534,13492,4,4,1681,2639.0
444,tipo_procedura,1,l’esercizio 2020. \n10. I PAGAMENTI: avvengono...,6.634639e-21,2489,2983,1,1,2489,2983.0
445,tipo_procedura,29,art. 76 del Codice a mezzo posta e/o fax o pos...,2.153163e-21,32825,33667,8,9,4112,NaN


# 29. Valore principale per la sidebar

Per ogni field mostriamo il candidato con confidence maggiore. Le alternative restano disponibili nei dati. Se esistono più valori distinti, lo stato diventa `AMBIGUOUS`; se non troviamo nulla, `NOT_FOUND`.

In [25]:
def select_field_values(candidates_df: pd.DataFrame, schema: dict) -> pd.DataFrame:
    rows=[]
    for field_name,cfg in schema["fields"].items():
        group=candidates_df[candidates_df["field"]==field_name].copy()
        if group.empty:
            rows.append({"field":field_name,"description":cfg["description"],"status":"NOT_FOUND","value":None,"confidence":None,"page":None,"global_start":None,"global_end":None,"alternative_count":0})
            continue
        group["_c"]=pd.to_numeric(group["confidence"],errors="coerce").fillna(-1.0)
        group=group.sort_values("_c",ascending=False)
        selected=group.iloc[0]
        distinct={re.sub(r"\s+"," ",str(v).lower()).strip() for v in group["text"].dropna()}
        rows.append({
            "field":field_name,"description":cfg["description"],
            "status":"AMBIGUOUS" if len(distinct)>1 else "EXTRACTED",
            "value":selected["text"],"confidence":selected["confidence"],"page":selected["start_page"],
            "global_start":selected["global_start"],"global_end":selected["global_end"],
            "alternative_count":max(0,len(distinct)-1),
        })
    return pd.DataFrame(rows)
EXTRACTED_FIELDS = select_field_values(FIELD_CANDIDATES, ACTIVE_SCHEMA)
display(EXTRACTED_FIELDS)

,field,description,status,value,confidence,page,global_start,global_end,alternative_count
0,stazione_appaltante,"Ente, amministrazione o organizzazione che ind...",AMBIGUOUS,S.O.A.,0.947538,5,19422,19428,11
1,oggetto_gara,"Oggetto principale della procedura, cioè la fo...",AMBIGUOUS,lavori di straordinaria manutenzione per la re...,0.497931,4,11226,11386,26
2,cig,Codice Identificativo Gara (CIG),AMBIGUOUS,84126312B8,0.665344,4,11537,11547,20
3,cup,"Codice Unico di Progetto (CUP), se presente",AMBIGUOUS,D75I14000050001,0.911220,1,929,944,28
4,cpv,"Codice CPV principale della procedura, se pres...",AMBIGUOUS,30125,0.443260,1,193,198,29
5,tipo_procedura,"Tipo di procedura di affidamento o gara, per e...",AMBIGUOUS,procedura negoziata,0.577097,1,623,642,27
6,importo_complessivo,Valore economico complessivo della procedura o...,AMBIGUOUS,"euro 149.866,23",0.619153,4,11409,11424,28
7,importo_base_asta,Importo posto a base d'asta o base di gara,AMBIGUOUS,"euro 145.866,23",0.802055,4,11433,11448,28
8,numero_lotti,Numero complessivo dei lotti in cui è suddivis...,AMBIGUOUS,4,0.411992,4,11476,11477,27
9,durata_contratto,"Durata prevista del contratto, appalto o servizio",AMBIGUOUS,provvisoria,0.910294,4,12302,12313,29


# 30. Verifica span dei field

In [26]:
field_checks=[]
for _,r in FIELD_CANDIDATES.iterrows():
    s,e=r["global_start"],r["global_end"]
    valid=None if pd.isna(s) or pd.isna(e) else DOCUMENT["full_text"][int(s):int(e)] == r["text"]
    field_checks.append({"field":r["field"],"text":r["text"],"page":r["start_page"],"span_valid":valid})
FIELD_SPAN_CHECK=pd.DataFrame(field_checks)
display(FIELD_SPAN_CHECK)

,field,text,page,span_valid
0,cig,84126312B8,4,True
1,cig,84126312B8,1,True
2,cig,CIG 84126312B8,4,True
3,cig,legale \nrappresentante dell’Impresa mandatari...,7,True
4,cig,Amministrazione,8,True
...,...,...,...,...
442,tipo_procedura,art. 76 del Codice a mezzo posta e/o fax o pos...,8,True
443,tipo_procedura,Amministrazione ad esito dell’esperimento dell...,4,True
444,tipo_procedura,l’esercizio 2020. \n10. I PAGAMENTI: avvengono...,1,True
445,tipo_procedura,art. 76 del Codice a mezzo posta e/o fax o pos...,8,True


# 31. Prototipo del viewer annotato

Nel documento mostriamo **solo la NER**. La Field Extraction rimane separata nella sidebar.

In [27]:
def schema_color(label:str,schema:dict)->str:
    return schema.get("entities",{}).get(label,{}).get("color","#FFF3BF")

def render_annotated_page(page_number:int,entities_df:pd.DataFrame,schema:dict)->HTML:
    page=DOCUMENT["pages"][page_number-1]; text=page["text"]
    ent=entities_df[(entities_df["start_page"]==page_number)&(entities_df["end_page"]==page_number)&entities_df["page_start"].notna()&entities_df["page_end"].notna()].copy().sort_values("page_start")
    parts=[]; cursor=0
    for _,e in ent.iterrows():
        s,end=int(e["page_start"]),int(e["page_end"])
        if s < cursor or s < 0 or end > len(text): continue
        parts.append(html.escape(text[cursor:s]))
        color=schema_color(e["label"],schema)
        parts.append(f'<span style="background:{color};padding:2px 4px;border-radius:4px;border:1px solid rgba(0,0,0,.12);line-height:1.8;">{html.escape(text[s:end])}<sup style="font-size:9px;margin-left:3px;font-weight:600;">{html.escape(str(e["label"]))}</sup></span>')
        cursor=end
    parts.append(html.escape(text[cursor:]))
    body="".join(parts).replace("\n","<br>")
    return HTML(f'<div style="border:1px solid #ddd;border-radius:10px;padding:22px;font-family:Arial;font-size:14px;line-height:1.7;"><div style="display:flex;justify-content:space-between;border-bottom:1px solid #eee;padding-bottom:10px;margin-bottom:18px;"><strong>{html.escape(DOCUMENT["filename"])}</strong><span>Pagina {page_number}</span></div>{body}</div>')

## Scegli una pagina

In [28]:
VIEWER_PAGE = 1
display(render_annotated_page(VIEWER_PAGE, VIEWER_ENTITIES, ACTIVE_SCHEMA))

# 32. Sidebar NER

In [29]:
def build_entity_sidebar(entities_df:pd.DataFrame,schema:dict)->pd.DataFrame:
    counts=entities_df.groupby("label").size().to_dict() if not entities_df.empty else {}
    return pd.DataFrame([{"enabled":True,"label":label,"color":cfg.get("color"),"occurrences":counts.get(label,0)} for label,cfg in schema["entities"].items()])
ENTITY_SIDEBAR=build_entity_sidebar(VIEWER_ENTITIES,ACTIVE_SCHEMA)
display(ENTITY_SIDEBAR)

,enabled,label,color,occurrences
0,True,ORGANIZZAZIONE,#DCEBFF,55
1,True,PERSONA,#FFE5D9,24
2,True,LOCALITA,#E1F5E5,12
3,True,IMPORTO,#FFF1B8,16
4,True,DATA,#E8DDF8,8
5,True,ORA,#F7D9EE,4
6,True,RIFERIMENTO_NORMATIVO,#D9F0F0,53
7,True,IDENTIFICATIVO_GARA,#FCE0B2,10
8,True,CERTIFICAZIONE,#E5E5FF,3
9,True,PRODOTTO_SERVIZIO,#DFF3FF,6


# 33. Sidebar Field Extraction

In [30]:
FIELD_SIDEBAR = EXTRACTED_FIELDS[["field","status","value","page","confidence","alternative_count"]].copy()
display(FIELD_SIDEBAR)

,field,status,value,page,confidence,alternative_count
0,stazione_appaltante,AMBIGUOUS,S.O.A.,5,0.947538,11
1,oggetto_gara,AMBIGUOUS,lavori di straordinaria manutenzione per la re...,4,0.497931,26
2,cig,AMBIGUOUS,84126312B8,4,0.665344,20
3,cup,AMBIGUOUS,D75I14000050001,1,0.911220,28
4,cpv,AMBIGUOUS,30125,1,0.443260,29
5,tipo_procedura,AMBIGUOUS,procedura negoziata,1,0.577097,27
6,importo_complessivo,AMBIGUOUS,"euro 149.866,23",4,0.619153,28
7,importo_base_asta,AMBIGUOUS,"euro 145.866,23",4,0.802055,28
8,numero_lotti,AMBIGUOUS,4,4,0.411992,27
9,durata_contratto,AMBIGUOUS,provvisoria,4,0.910294,29


# 34. Mock combinato documento + sidebar

In [31]:
def render_documind_prototype(page_number:int,entities_df:pd.DataFrame,fields_df:pd.DataFrame,schema:dict)->HTML:
    page=DOCUMENT["pages"][page_number-1]; text=page["text"]
    ent=entities_df[(entities_df["start_page"]==page_number)&(entities_df["end_page"]==page_number)&entities_df["page_start"].notna()&entities_df["page_end"].notna()].copy().sort_values("page_start")
    parts=[]; cursor=0
    for _,e in ent.iterrows():
        s,end=int(e["page_start"]),int(e["page_end"])
        if s < cursor or s < 0 or end > len(text): continue
        parts.append(html.escape(text[cursor:s]))
        color=schema_color(e["label"],schema)
        parts.append(f'<span style="background:{color};padding:2px 4px;border-radius:4px;">{html.escape(text[s:end])}<sup style="font-size:9px;font-weight:600;"> {html.escape(str(e["label"]))}</sup></span>')
        cursor=end
    parts.append(html.escape(text[cursor:]))
    doc_html="".join(parts).replace("\n","<br>")
    counts=entities_df.groupby("label").size().to_dict() if not entities_df.empty else {}
    entity_rows=[]
    for label,cfg in schema["entities"].items():
        entity_rows.append(f'<div style="margin-bottom:8px;"><span style="display:inline-block;width:12px;height:12px;background:{cfg.get("color","#eee")};border:1px solid #bbb;border-radius:3px;margin-right:6px;"></span><strong>{html.escape(label)}</strong><span style="float:right;">{counts.get(label,0)}</span></div>')
    field_rows=[]
    for _,f in fields_df.iterrows():
        value=html.escape(str(f["value"])) if pd.notna(f["value"]) else "Non trovato"
        page_label=f'pag. {int(f["page"])}' if pd.notna(f["page"]) else ""
        field_rows.append(f'<div style="padding:10px 0;border-bottom:1px solid #eee;"><div style="font-size:12px;color:#666;">{html.escape(str(f["field"]))}</div><div style="font-weight:600;margin-top:2px;">{value}</div><div style="font-size:11px;color:#777;margin-top:3px;">{html.escape(str(f["status"]))} &nbsp; {page_label}</div></div>')
    return HTML(f'<div style="display:grid;grid-template-columns:minmax(0,2.3fr) minmax(260px,.7fr);gap:18px;font-family:Arial;"><div style="border:1px solid #ddd;border-radius:10px;background:white;overflow:hidden;"><div style="padding:14px 18px;border-bottom:1px solid #eee;display:flex;justify-content:space-between;"><strong>{html.escape(DOCUMENT["filename"])}</strong><span>Pagina {page_number}</span></div><div style="padding:24px;line-height:1.8;font-size:14px;">{doc_html}</div></div><div style="border:1px solid #ddd;border-radius:10px;background:#fafafa;padding:16px;"><h3 style="margin-top:0;">Entità</h3>{"".join(entity_rows)}<h3 style="margin-top:24px;padding-top:16px;border-top:1px solid #ddd;">Informazioni estratte</h3>{"".join(field_rows)}</div></div>')

display(render_documind_prototype(VIEWER_PAGE,VIEWER_ENTITIES,EXTRACTED_FIELDS,ACTIVE_SCHEMA))

# 35. Orchestratore completo

Questa funzione rappresenta il comportamento futuro del servizio di analisi del backend. Riceve `pdf_path`, `schema` e configurazione e restituisce le strutture dati pronte per essere persistite.

In [32]:
def analyze_pdf(pdf_path:Path,schema:dict,config:dict=PIPELINE_CONFIG)->dict:
    document=load_pdf_with_page_map(pdf_path)
    if document["chars"] < 50: raise ValueError("PDF con troppo poco testo estraibile")
    entity_types=build_entity_types(schema)
    t=time.perf_counter()
    raw_entity_result=EXTRACTOR.extract_entities_long(document["full_text"],entity_types,threshold=config["entity_threshold"],chunk_size=config["chunk_size"],chunk_overlap=config["chunk_overlap"],batch_size=config["batch_size"],include_spans=True,include_confidence=True,overlap_policy=config["model_overlap_policy"])
    entity_seconds=time.perf_counter()-t
    raw_entities=normalize_entity_result(raw_entity_result,document)
    viewer_entities=resolve_overlaps_for_viewer(deduplicate_entities(raw_entities))
    json_structure=build_json_structure(schema)
    t=time.perf_counter()
    raw_field_result=EXTRACTOR.extract_json_long(document["full_text"],json_structure,threshold=config["field_threshold"],chunk_size=config["chunk_size"],chunk_overlap=config["chunk_overlap"],batch_size=config["batch_size"],include_spans=True,include_confidence=True,overlap_policy=config["model_overlap_policy"])
    field_seconds=time.perf_counter()-t
    field_candidates=deduplicate_field_candidates(normalize_field_result(raw_field_result,schema,document))
    extracted_fields=select_field_values(field_candidates,schema)
    return {
        "document":document,"schema_name":schema["name"],"config":config,
        "raw_entities":raw_entities,"viewer_entities":viewer_entities,
        "field_candidates":field_candidates,"extracted_fields":extracted_fields,
        "metrics":{"entity_seconds":entity_seconds,"field_seconds":field_seconds,"total_ai_seconds":entity_seconds+field_seconds,"raw_entity_count":len(raw_entities),"viewer_entity_count":len(viewer_entities),"field_candidate_count":len(field_candidates),"extracted_field_count":int((extracted_fields["status"]!="NOT_FOUND").sum())}
    }